# 面试问题：KV Cache 低比特量化、分组轴与 Residual Window 怎样实现？

可直接复述的回答：KV Cache 随层数、序列和并发线性增长，因此低比特量化能直接扩大服务容量。K 常在通道维存在稳定离群值，适合 per-channel qparam；V 随 token 变化更明显，常用 per-token qparam，但必须按目标模型校准。2-bit 值需要真正 packing 才能获得内存收益，int8 容器本身不是 2-bit 存储。最近 token 可保留高精度 residual window，降低解码最敏感部分的误差。评估应比较 attention 输出、任务质量和延迟，而不只看张量 MSE。异常值、动态范围和 kernel ABI 都要纳入发布合同。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：客服对话 KV 与输入预览

八个 token 来自退款对话，K/V 为单层单头四维教学缓存。K 的第三通道包含稳定大值，V 的每个 token 幅度不同，用于展示分组轴选择。


In [1]:
import numpy as np  # 使用 NumPy 手写低比特量化与 attention。
tokens04 = ["用户", "申请", "订单", "退款", "工具", "校验", "审批", "完成"]  # 构造有顺序语义的缓存 token。
k04 = np.array([[0.2, -0.4, 7.0, 0.1], [0.3, -0.2, 6.8, 0.2], [0.5, -0.1, 7.2, 0.3], [0.4, 0.2, 6.9, 0.5], [0.6, 0.4, 7.1, 0.7], [0.7, 0.5, 7.3, 0.8], [0.9, 0.7, 6.7, 1.0], [1.0, 0.9, 7.0, 1.2]], dtype=np.float64)  # 构造带通道离群值的 K。
v04 = np.array([[0.1, 0.2, -0.1, 0.0], [0.2, 0.4, -0.2, 0.1], [0.4, 0.8, -0.3, 0.2], [0.7, 1.1, -0.5, 0.3], [1.0, 1.5, -0.8, 0.5], [1.4, 2.0, -1.0, 0.7], [1.8, 2.6, -1.3, 0.9], [2.2, 3.1, -1.6, 1.1]], dtype=np.float64)  # 构造随 token 幅度变化的 V。
query04 = np.array([0.3, 0.1, 0.25, 0.2], dtype=np.float64)  # 定义下一 token 的查询向量。
print("教学实验输入：position | token | K | V")  # 输出输入预览表头。
for position04, token04 in enumerate(tokens04):  # 逐 token 展示缓存数值。
    print(position04, token04, k04[position04].tolist(), v04[position04].tolist())  # 输出一行 KV 记录。


教学实验输入：position | token | K | V
0 用户 [0.2, -0.4, 7.0, 0.1] [0.1, 0.2, -0.1, 0.0]
1 申请 [0.3, -0.2, 6.8, 0.2] [0.2, 0.4, -0.2, 0.1]
2 订单 [0.5, -0.1, 7.2, 0.3] [0.4, 0.8, -0.3, 0.2]
3 退款 [0.4, 0.2, 6.9, 0.5] [0.7, 1.1, -0.5, 0.3]
4 工具 [0.6, 0.4, 7.1, 0.7] [1.0, 1.5, -0.8, 0.5]
5 校验 [0.7, 0.5, 7.3, 0.8] [1.4, 2.0, -1.0, 0.7]
6 审批 [0.9, 0.7, 6.7, 1.0] [1.8, 2.6, -1.3, 0.9]
7 完成 [1.0, 0.9, 7.0, 1.2] [2.2, 3.1, -1.6, 1.1]


## 2. Baseline（基线）：整个 K/V 各用一组 2-bit QParam

朴素 per-tensor 非对称量化用 0–3 四个 code 表示整个矩阵。K 的大通道会压缩其他通道，V 的后期大值会损伤早期 token。


In [2]:
def quantize2_04(values04, axis04=None):  # 实现四级非对称量化。
    minimum04 = values04.min(axis=axis04, keepdims=True)  # 计算指定分组的最小值。
    maximum04 = values04.max(axis=axis04, keepdims=True)  # 计算指定分组的最大值。
    scale04 = np.maximum((maximum04 - minimum04) / 3.0, 1e-8)  # 把动态范围映射到四个 code。
    codes04 = np.round((values04 - minimum04) / scale04).clip(0, 3).astype(np.uint8)  # 量化为 0到3 的 2-bit code。
    restored04 = codes04.astype(np.float64) * scale04 + minimum04  # 反量化回浮点缓存。
    return codes04, restored04, minimum04, scale04  # 返回 code、恢复值和 qparam。
k_codes_base04, k_base04, k_min_base04, k_scale_base04 = quantize2_04(k04, None)  # 对整个 K 使用单组 qparam。
v_codes_base04, v_base04, v_min_base04, v_scale_base04 = quantize2_04(v04, None)  # 对整个 V 使用单组 qparam。
k_mse_base04 = float(np.mean((k_base04 - k04) ** 2))  # 计算 K 基线量化误差。
v_mse_base04 = float(np.mean((v_base04 - v04) ** 2))  # 计算 V 基线量化误差。
print("Per-tensor基线QParam", {"K_min": float(k_min_base04.item()), "K_scale": float(k_scale_base04.item()), "V_min": float(v_min_base04.item()), "V_scale": float(v_scale_base04.item())})  # 展示粗粒度量化参数。
print("Per-tensor MSE", {"K": round(k_mse_base04, 5), "V": round(v_mse_base04, 5)})  # 展示缓存张量误差。


Per-tensor基线QParam {'K_min': -0.4, 'K_scale': 2.566666666666667, 'V_min': -1.6, 'V_scale': 1.5666666666666667}
Per-tensor MSE {'K': 0.62767, 'V': 0.18771}


## 3. 核心实现：K Per-channel、V Per-token 与 2-bit Packing

K 沿 token 轴统计每个通道 qparam，V 沿 hidden 轴统计每个 token qparam。四个 2-bit code 打包进一个 byte，下面同时展示 qparam 形状与打包字节。


In [3]:
def pack2_04(codes04):  # 把四个 2-bit code 打包成一个字节。
    flat04 = codes04.reshape(-1)  # 展平量化 code 便于顺序打包。
    padding04 = (-len(flat04)) % 4  # 计算凑齐四元素分组所需 padding。
    padded04 = np.pad(flat04, (0, padding04))  # 在尾部补零 code。
    groups04 = padded04.reshape(-1, 4).astype(np.uint8)  # 每四个 code 组成一组。
    packed04 = groups04[:, 0] | (groups04[:, 1] << 2) | (groups04[:, 2] << 4) | (groups04[:, 3] << 6)  # 使用位移合并四个 code。
    return packed04, padding04  # 返回真实打包字节和 padding 数。
k_codes_core04, k_core04, k_min_core04, k_scale_core04 = quantize2_04(k04, axis04=0)  # 对 K 每通道量化。
v_codes_core04, v_core04, v_min_core04, v_scale_core04 = quantize2_04(v04, axis04=1)  # 对 V 每 token 量化。
residual_window04 = 2  # 设置最近两个 token 保留高精度。
k_core04[-residual_window04:] = k04[-residual_window04:]  # 恢复最近 K 为高精度值。
v_core04[-residual_window04:] = v04[-residual_window04:]  # 恢复最近 V 为高精度值。
packed_k04, k_padding04 = pack2_04(k_codes_core04)  # 真正打包 K 的 2-bit code。
packed_v04, v_padding04 = pack2_04(v_codes_core04)  # 真正打包 V 的 2-bit code。
print("核心QParam形状", {"K_per_channel_min": k_min_core04.shape, "K_scale": k_scale_core04.shape, "V_per_token_min": v_min_core04.shape, "V_scale": v_scale_core04.shape})  # 展示不同分组轴。
print("2-bit packing", {"original_codes": int(k_codes_core04.size + v_codes_core04.size), "packed_bytes": int(packed_k04.size + packed_v04.size), "padding": int(k_padding04 + v_padding04), "first_K_bytes": packed_k04[:4].tolist()})  # 展示真实字节压缩。


核心QParam形状 {'K_per_channel_min': (1, 4), 'K_scale': (1, 4), 'V_per_token_min': (8, 1), 'V_scale': (8, 1)}
2-bit packing {'original_codes': 64, 'packed_bytes': 16, 'padding': 0, 'first_K_bytes': [32, 0, 117, 85]}


## 4. 结果表、Attention 误差与结果解读

最终目标不是 K/V 自身 MSE，而是 attention 输出。这里用同一 query 比较 FP64 oracle、per-tensor 和分轴+residual window 的上下文向量。


In [4]:
def attention_output04(query04, keys04, values04):  # 计算单 query attention 输出。
    scores04 = query04 @ keys04.T / np.sqrt(keys04.shape[1])  # 计算缩放点积得分。
    weights04 = np.exp(scores04 - scores04.max())  # 计算稳定 softmax 分子。
    weights04 = weights04 / weights04.sum()  # 归一化全部历史 token 权重。
    return weights04 @ values04, weights04  # 返回上下文向量与注意力权重。
oracle_output04, oracle_weights04 = attention_output04(query04, k04, v04)  # 计算高精度 attention oracle。
baseline_output04, _ = attention_output04(query04, k_base04, v_base04)  # 计算 per-tensor 量化输出。
core_output04, core_weights04 = attention_output04(query04, k_core04, v_core04)  # 计算分轴加 residual 输出。
baseline_error04 = float(np.linalg.norm(baseline_output04 - oracle_output04))  # 计算基线 attention 输出误差。
core_error04 = float(np.linalg.norm(core_output04 - oracle_output04))  # 计算核心方案 attention 输出误差。
print("方法 | K_MSE | V_MSE | attention_output_L2 | context")  # 输出结果对照表头。
print("per_tensor", round(k_mse_base04, 5), round(v_mse_base04, 5), round(baseline_error04, 6), np.round(baseline_output04, 4).tolist())  # 展示粗粒度量化结果。
print("axis_plus_residual", round(float(np.mean((k_core04 - k04) ** 2)), 5), round(float(np.mean((v_core04 - v04) ** 2)), 5), round(core_error04, 6), np.round(core_output04, 4).tolist())  # 展示分轴量化结果。
print("结果解读：分组轴保护K离群通道，最近高精度窗口进一步降低解码敏感误差")  # 解释数值改进来源。


方法 | K_MSE | V_MSE | attention_output_L2 | context
per_tensor 0.62767 0.18771 0.342062 [0.9104, 1.8541, -0.8213, 0.5989]
axis_plus_residual 0.00667 0.01542 0.140031 [0.9429, 1.5568, -0.7741, 0.6078]
结果解读：分组轴保护K离群通道，最近高精度窗口进一步降低解码敏感误差


## 5. 失败案例与修正：只看压缩率忽略最近 Token

最近 token 往往获得较高注意力权重。若全部量化，最后两步误差会直接进入下一 token；修正是小型 residual window，并监控其额外字节。


In [5]:
_, k_all_quant04, _, _ = quantize2_04(k04, axis04=0)  # 构造没有 residual window 的 K。
_, v_all_quant04, _, _ = quantize2_04(v04, axis04=1)  # 构造没有 residual window 的 V。
all_quant_output04, _ = attention_output04(query04, k_all_quant04, v_all_quant04)  # 计算全量低比特 attention 输出。
all_quant_error04 = float(np.linalg.norm(all_quant_output04 - oracle_output04))  # 计算无窗口输出误差。
residual_extra_bytes04 = residual_window04 * k04.shape[1] * 2 * 8  # 估算 FP64 教学窗口的额外 K/V 字节。
print("失败行为：全部2-bit attention误差", round(all_quant_error04, 6))  # 展示最新 token 也被量化的误差。
print("修正行为：保留最近", residual_window04, "token，误差", round(core_error04, 6), "额外教学字节", residual_extra_bytes04)  # 展示误差与内存权衡。


失败行为：全部2-bit attention误差 0.338988
修正行为：保留最近 2 token，误差 0.140031 额外教学字节 128


## 6. 生产边界与缓存制品

真实系统要按层/头校准轴与 group size，并使用 GPU pack/dequant kernel。还需处理 paged KV、prefix cache、beam/speculative token、残差窗口迁移和 kernel ABI。


In [6]:
kv_contract04 = {"bits": 2, "K_qparam": "per_channel", "V_qparam": "per_token", "residual_window": residual_window04, "packing": "4_codes_per_byte", "calibration": "target_model_layerwise"}  # 定义低比特缓存发布合同。
print("KV量化制品", kv_contract04)  # 展示位宽、轴、窗口和校准范围。
print("生产替换点：GPU bit-pack kernel、paged KV、逐层校准、prefix cache兼容和任务质量门禁")  # 说明 NumPy 教学实现的边界。


KV量化制品 {'bits': 2, 'K_qparam': 'per_channel', 'V_qparam': 'per_token', 'residual_window': 2, 'packing': '4_codes_per_byte', 'calibration': 'target_model_layerwise'}
生产替换点：GPU bit-pack kernel、paged KV、逐层校准、prefix cache兼容和任务质量门禁


## 7. 最小回归测试

断言保护真实 packing、分组轴和 attention 误差收益。


In [7]:
assert len(tokens04) >= 5  # 保证案例包含足够长的 KV 序列。
assert packed_k04.size * 4 >= k_codes_core04.size  # 保证四个 2-bit code 被真实打包进字节。
assert k_min_core04.shape == (1, k04.shape[1])  # 保证 K 使用 per-channel qparam。
assert v_min_core04.shape == (v04.shape[0], 1)  # 保证 V 使用 per-token qparam。
assert core_error04 < baseline_error04  # 保证分轴与 residual window 改善 attention 输出。
assert core_error04 <= all_quant_error04  # 保证最近高精度窗口不劣于全部量化。
print("最小回归测试通过：2-bit packing、QParam轴和Residual Window稳定")  # 显示 KV 量化关键性质已验证。


最小回归测试通过：2-bit packing、QParam轴和Residual Window稳定
